# WaveNet GAN for Synthetic Financial Data Generation

This notebook implements a **WaveNet-based GAN** adapted from the swim accelerometer/gyroscope WaveNet GAN architecture,
repurposed for unconditional financial time-series generation on SP500 log returns.

### Architecture Overview
- **Generator**: Latent noise → WaveNet block (dilated causal convolutions with gated activations) → Dense output
- **Discriminator**: Real/Fake data → WaveNet block → per-timestep real/fake classification
- **Training**: Adversarial + Huber reconstruction + **moment matching** + **tail quantile matching** loss, gradient clipping, adaptive LR scheduling

### Key differences from LSTM TimeGAN (notebook 03)
| Aspect | LSTM TimeGAN | WaveNet GAN |
|--------|-------------|-------------|
| Core unit | LSTM (recurrent) | Dilated causal convolutions |
| Activations | sigmoid | Gated: tanh ⊙ sigmoid |
| Receptive field | Sequential (slow) | Exponential (2^n dilations) |
| Skip connections | None | Sum of all dilation layers |
| Parameters | ~33% more (4 gates) | Fewer, parallelizable |

## 1. Load and Prepare Data

In [1]:
# Import libraries
import os
import site
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

# Suppress TF C++ info/warning spam (must be set BEFORE importing TF)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# XLA fix for pip-bundled CUDA toolkit
_cuda_nvcc_dir = os.path.join(
    site.getsitepackages()[0] if site.getsitepackages() else
    os.path.join(os.path.dirname(os.__file__), '..', 'site-packages'),
    'nvidia', 'cuda_nvcc'
)
os.environ['XLA_FLAGS'] = f'--xla_gpu_cuda_data_dir={_cuda_nvcc_dir}'

import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Conv1D, Multiply, Add, Concatenate, Dense,
    Dropout, GaussianNoise, LayerNormalization
)
from tensorflow.keras.models import Model, load_model

# Set seeds
tf.random.set_seed(42)
np.random.seed(42)

# Suppress Python-level TF logs
tf.get_logger().setLevel('ERROR')

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

TensorFlow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
# Load SP500 data
data = pd.read_csv('../data/raw/sp500.csv', index_col='Date', parse_dates=True)
data = data.apply(pd.to_numeric, errors='coerce')

close_prices = data['Close']
log_returns = np.log(close_prices / close_prices.shift(1)).dropna()
log_returns_array = log_returns.values.reshape(-1, 1)

# Normalize to [0, 1]
scaler = MinMaxScaler()
log_returns_scaled = scaler.fit_transform(log_returns_array)

# Create sequences
sequence_length = 24  # Match LSTM TimeGAN for fair comparison
sequences = []
for i in range(len(log_returns_scaled) - sequence_length):
    sequences.append(log_returns_scaled[i:i+sequence_length])

sequences = np.array(sequences, dtype=np.float32)
print(f"Data shape: {sequences.shape}")
print(f"Log returns range: [{log_returns.min():.4f}, {log_returns.max():.4f}]")
print(f"Scaled range: [{sequences.min():.4f}, {sequences.max():.4f}]")

Data shape: (6012, 24, 1)
Log returns range: [-0.1277, 0.1096]
Scaled range: [0.0000, 1.0000]


## 2. WaveNet Building Blocks

Adapted from `wavenet_gan_uncond_noise.py` — gated causal convolutions with skip connections.

Key adaptations for financial data:
- Reduced `nfilt` (32 vs 64) since we have 1 feature vs 6 sensors
- Reduced dilation rates `[1,2,4,8,16]` since `seq_len=24` vs 180
- No conditional inputs (style/stroke) — unconditional generation

In [3]:
###############################################
# WaveNet Residual Block with Optional Gaussian Noise
# Adapted from: wavenet_gan_uncond_noise.py
###############################################
def wavenet_residual_block(input_tensor, nfilt, dilation_rate, 
                           residual_noise_std=None, seed=None):
    """Gated causal convolution block with skip connection.
    
    The core building block: uses tanh * sigmoid gating (like the 
    original WaveNet paper) instead of plain activations.
    """
    x = input_tensor
    
    # Project to nfilt channels if needed
    if x.shape[-1] != nfilt:
        x = Conv1D(filters=nfilt, kernel_size=1, padding='same')(x)
    
    # Optional noise injection for regularization
    if residual_noise_std:
        x = GaussianNoise(stddev=residual_noise_std, seed=seed)(x)
    
    # Gated activation: tanh(conv) * sigmoid(conv)
    tanh_out = Conv1D(filters=nfilt, kernel_size=3, dilation_rate=dilation_rate,
                      padding='causal', activation='tanh')(x)
    sigm_out = Conv1D(filters=nfilt, kernel_size=3, dilation_rate=dilation_rate,
                      padding='causal', activation='sigmoid')(x)
    gated = Multiply()([tanh_out, sigm_out])
    
    # Skip and residual outputs
    skip_out = Conv1D(filters=nfilt, kernel_size=1, padding='same')(gated)
    residual = Conv1D(filters=nfilt, kernel_size=1, padding='same')(gated)
    residual_out = Add()([x, residual])
    
    return residual_out, skip_out


###############################################
# WaveNet Block — stack of dilated residual blocks
###############################################
def wavenet_block(input_tensor, nfilt, dilation_rates=None,
                  residual_noise_std=None, seed=None):
    """Stack of dilated causal convolutions with exponentially increasing dilation.
    
    Receptive field = sum(dilation_rates) * (kernel_size - 1) + 1
    For [1,2,4,8,16]: receptive field = 31 * 2 + 1 = 63 (covers seq_len=24)
    """
    if dilation_rates is None:
        dilation_rates = [1, 2, 4, 8, 16]  # Reduced for seq_len=24
    
    skip_connections = []
    x = input_tensor
    for i, dilation in enumerate(dilation_rates):
        x, skip = wavenet_residual_block(
            x, nfilt, dilation,
            residual_noise_std=residual_noise_std,
            seed=(seed + i) if seed else None
        )
        skip_connections.append(skip)
    return Add()(skip_connections)


###############################################
# Deep WaveNet — multiple WaveNet blocks stacked
###############################################
def deep_wavenet(input_tensor, nfilt, n_stacks=2,
                 residual_noise_std=None, seed=None):
    """Stack multiple WaveNet blocks for deeper temporal modeling."""
    x = input_tensor
    for i in range(n_stacks):
        x = wavenet_block(
            x, nfilt,
            residual_noise_std=residual_noise_std,
            seed=(seed + 100 + i) if seed else None
        )
    return x


print("WaveNet building blocks defined.")

WaveNet building blocks defined.


## 3. Build Generator and Discriminator

**Generator**: Noise → WaveNet → financial sequences  
**Discriminator**: Sequences → WaveNet → per-timestep real/fake score

In [4]:
###############################################
# Generator
###############################################
def build_wavenet_generator(sequence_length=24, feature_dim=1, latent_dim=32, nfilt=32,
                            n_stacks=2, latent_noise_std=0.01, 
                            residual_noise_std=0.01, seed=None):
    """WaveNet-based generator for unconditional financial time-series.
    
    Adapted from build_swim_generator in wavenet_gan_uncond_noise.py,
    but without conditional inputs (style/stroke labels).
    """
    latent_in = Input(shape=(sequence_length, latent_dim), name='latent_input')
    
    x = latent_in
    
    # Latent augmentation (from swim repo generator)
    if latent_noise_std:
        x = GaussianNoise(latent_noise_std, name='latent_noise',
                          seed=seed)(x)
    
    # Process through WaveNet
    x = deep_wavenet(x, nfilt, n_stacks=n_stacks,
                     residual_noise_std=residual_noise_std, seed=seed)
    
    # Project to feature dimension with sigmoid (output in [0,1] to match MinMaxScaled data)
    output = Dense(feature_dim, activation='sigmoid', name='generator_output')(x)
    
    model = Model(inputs=latent_in, outputs=output, name='WaveNet_Generator')
    return model


###############################################
# Discriminator
###############################################
def build_wavenet_discriminator(sequence_length=24, feature_dim=1, nfilt=32,
                                n_stacks=2, residual_noise_std=None,
                                dropout_rate=None, seed=None):
    """WaveNet-based discriminator for financial time-series.
    
    Adapted from build_swim_discriminator in wavenet_gan_uncond_noise.py.
    Per-timestep classification (real/fake at each time step).
    """
    data_in = Input(shape=(sequence_length, feature_dim), name='data_input')
    
    x = deep_wavenet(data_in, nfilt, n_stacks=n_stacks,
                     residual_noise_std=residual_noise_std, seed=seed)
    
    # Optional dropout
    if dropout_rate:
        x = Dropout(dropout_rate, seed=seed)(x)
    
    # Per-timestep real/fake output
    output = Dense(1, activation='sigmoid', name='real_fake_output')(x)
    
    model = Model(inputs=data_in, outputs=output, name='WaveNet_Discriminator')
    return model


print("Generator and Discriminator builders defined.")

Generator and Discriminator builders defined.


## 4. Instantiate Models and Training Setup

In [5]:
# Hyperparameters
LATENT_DIM = 32        # Noise vector dimension per timestep
NFILT = 32             # WaveNet filter count
N_STACKS = 2           # Number of stacked WaveNet blocks
FEATURE_DIM = 1        # Single feature (log return)
SEQUENCE_LENGTH = 24   # Match LSTM TimeGAN

LR_GENERATOR = 0.0002
LR_DISCRIMINATOR = 0.0001  # Lower disc LR for stability (from swim repo)
BETA_1 = 0.5
BATCH_SIZE = 128
EPOCHS = 2000

# Build models
generator = build_wavenet_generator(
    sequence_length=SEQUENCE_LENGTH,
    feature_dim=FEATURE_DIM,
    latent_dim=LATENT_DIM,
    nfilt=NFILT,
    n_stacks=N_STACKS,
    latent_noise_std=0.01,
    residual_noise_std=0.01,
    seed=42
)

discriminator = build_wavenet_discriminator(
    sequence_length=SEQUENCE_LENGTH,
    feature_dim=FEATURE_DIM,
    nfilt=NFILT,
    n_stacks=N_STACKS,
    residual_noise_std=0.01,
    dropout_rate=0.1,
    seed=42
)

# Optimizers (from swim repo: separate LRs, beta_1=0.5 for GAN stability)
gen_optimizer = tf.keras.optimizers.Adam(
    learning_rate=LR_GENERATOR, beta_1=BETA_1
)
disc_optimizer = tf.keras.optimizers.Adam(
    learning_rate=LR_DISCRIMINATOR, beta_1=BETA_1
)

generator.summary()
print()
discriminator.summary()

I0000 00:00:1772855647.819766 3972278 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1893 MB memory:  -> device: 0, name: Quadro P620, pci bus id: 0000:01:00.0, compute capability: 6.1


Model: "WaveNet_Generator"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ latent_input        │ (None, 24, 32)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ latent_noise        │ (None, 24, 32)    │          0 │ latent_input[0][… │
│ (GaussianNoise)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gaussian_noise      │ (None, 24, 32)    │          0 │ latent_noise[0][… │
│ (GaussianNoise)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 24, 32)    │      3,104 │ gaussian_noise[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 24, 32)    │      3,104 │ gaussian_noise[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 24, 32)    │          0 │ conv1d[0][0],     │
│                     │                   │            │ conv1d_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 24, 32)    │      1,056 │ multiply[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 24, 32)    │          0 │ gaussian_noise[0… │
│                     │                   │            │ conv1d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gaussian_noise_1    │ (None, 24, 32)    │          0 │ add[0][0]         │
│ (GaussianNoise)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 24, 32)    │      3,104 │ gaussian_noise_1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_5 (Conv1D)   │ (None, 24, 32)    │      3,104 │ gaussian_noise_1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_1          │ (None, 24, 32)    │          0 │ conv1d_4[0][0],   │
│ (Multiply)          │                   │            │ conv1d_5[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_7 (Conv1D)   │ (None, 24, 32)    │      1,056 │ multiply_1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 24, 32)    │          0 │ gaussian_noise_1… │
│                     │                   │            │ conv1d_7[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gaussian_noise_2    │ (None, 24, 32)    │          0 │ add_1[0][0]       │
│ (GaussianNoise)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_8 (Conv1D)   │ (None, 24, 32)    │      3,104 │ gaussian_noise_2… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_9 (Conv1D)   │ (None, 24, 32)    │      3,104 │ gaussian_noise_2… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_2          │ (None, 24, 32)    │          0 │ conv1d_8[0][0],   │
│ (Multiply)          │                   │            │ conv1d_9[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_11 (Conv1D)  │ (None, 24, 32)    │      1,056 │ multiply_2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 24, 32)    │          0 │ gaussian_noise_2

 Total params: 81,121 (316.88 KB)

 Trainable params: 81,121 (316.88 KB)

...

Model: "WaveNet_Discriminator"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ data_input          │ (None, 24, 1)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_40 (Conv1D)  │ (None, 24, 32)    │         64 │ data_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gaussian_noise_10   │ (None, 24, 32)    │          0 │ conv1d_40[0][0]   │
│ (GaussianNoise)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_41 (Conv1D)  │ (None, 24, 32)    │      3,104 │ gaussian_noise_1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_42 (Conv1D)  │ (None, 24, 32)    │      3,104 │ gaussian_noise_1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_10         │ (None, 24, 32)    │          0 │ conv1d_41[0][0],  │
│ (Multiply)          │                   │            │ conv1d_42[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_44 (Conv1D)  │ (None, 24, 32)    │      1,056 │ multiply_10[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_12 (Add)        │ (None, 24, 32)    │          0 │ gaussian_noise_1… │
│                     │                   │            │ conv1d_44[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gaussian_noise_11   │ (None, 24, 32)    │          0 │ add_12[0][0]      │
│ (GaussianNoise)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_45 (Conv1D)  │ (None, 24, 32)    │      3,104 │ gaussian_noise_1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_46 (Conv1D)  │ (None, 24, 32)    │      3,104 │ gaussian_noise_1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_11         │ (None, 24, 32)    │          0 │ conv1d_45[0][0],  │
│ (Multiply)          │                   │            │ conv1d_46[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_48 (Conv1D)  │ (None, 24, 32)    │      1,056 │ multiply_11[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_13 (Add)        │ (None, 24, 32)    │          0 │ gaussian_noise_1… │
│                     │                   │            │ conv1d_48[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gaussian_noise_12   │ (None, 24, 32)    │          0 │ add_13[0][0]      │
│ (GaussianNoise)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_49 (Conv1D)  │ (None, 24, 32)    │      3,104 │ gaussian_noise_1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_50 (Conv1D)  │ (None, 24, 32)    │      3,104 │ gaussian_noise_1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_12         │ (None, 24, 32)    │          0 │ conv1d_49[0][0],  │
│ (Multiply)          │                   │            │ conv1d_50[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_52 (Conv1D)  │ (None, 24, 32)    │      1,056 │ multiply_12[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_14 (Add)        │ (None, 24, 32)    │          0 │ gaussian_noise_1… │
│                     │                   │            │ conv1d_52[0][0] 

 Total params: 81,185 (317.13 KB)

 Trainable params: 81,185 (317.13 KB)

 Non-trainable params: 0 (0.00 B)

## 5. Training Loop

Adapted from `train_step6` in `wavenet_gan_train_uncond_noise.py`:  
- Adversarial loss (binary crossentropy with label smoothing from swim repo)  
- Huber reconstruction loss (from swim repo — more robust than MSE)  
- **Moment matching loss** (std + skewness + kurtosis penalties for tail fidelity)  
- **Tail quantile matching loss** (MSE on bottom/top 5% sorted values for extreme-value fidelity)  
- Gradient clipping to [-1, 1] (from swim repo)  
- Mini-batch training (unlike the LSTM TimeGAN which used full dataset per step)

In [6]:
import sys, datetime
sys.path.append('..')
from utils.models_utils import (smooth_positive_labels, smooth_negative_labels,
                                BalancedAdaptiveLearningRateSchedule,
                                compute_moment_loss, compute_tail_loss)

# Loss functions
bce = tf.keras.losses.BinaryCrossentropy()
huber = tf.keras.losses.Huber(delta=0.1)

# Lambda weights for generator loss components
LAMBDA_ADV = 1.0        # Adversarial loss weight
LAMBDA_RECON = 100.0    # Reconstruction loss weight (from swim repo lambda_g_real)
LAMBDA_MOMENT = 1.0     # Moment matching loss weight (std + skew + kurtosis)
LAMBDA_TAIL = 50.0      # Tail quantile matching loss weight (5% tails)

# ============================================================
# Balanced Adaptive LR — adjusts G/D rates to maintain equilibrium
# ============================================================
adaptive_lr = BalancedAdaptiveLearningRateSchedule(
    initial_gen_lr=LR_GENERATOR,
    initial_disc_lr=LR_DISCRIMINATOR,
    adjustment_factor=1.1,      # Gentler 10% steps (was 20%)
    tolerance=0.4,              # Equilibrium band [0.6, 1.4] — covers natural ~0.64 ratio
    min_lr=1e-5,                # Tighter floor (was 1e-6)
    max_lr=5e-4,                # Tighter ceiling (was 1e-2)
    max_lr_ratio=5.0,           # Max 5x ratio between disc/gen LRs
)
LR_ADJUST_EVERY = 50  # Re-balance LRs every N epochs

# ============================================================
# TensorBoard — log training metrics for monitoring
# ============================================================
log_dir = os.path.join('..', 'logs', 'wavenet_gan',
                       datetime.datetime.now().strftime('%Y%m%d-%H%M%S'))
tb_writer = tf.summary.create_file_writer(log_dir)

print(f"TensorBoard log dir: {log_dir}")
print(f"  → Launch: tensorboard --logdir {os.path.dirname(log_dir)}")


@tf.function
def train_step(real_data, batch_size):
    """Single training step adapted from train_step6 in wavenet_gan_train_uncond_noise.py."""
    
    # Generate noise (matching swim repo: normal distribution, not uniform)
    noise_gen = tf.random.normal(
        shape=(batch_size, SEQUENCE_LENGTH, LATENT_DIM), mean=0.0, stddev=1.0
    )
    noise_disc = tf.random.normal(
        shape=(batch_size, SEQUENCE_LENGTH, LATENT_DIM), mean=0.0, stddev=1.0
    )
    
    with tf.GradientTape() as disc_tape, tf.GradientTape() as gen_tape:
        # Generate fake data (separate noise for G and D — from swim repo)
        fake_data_gen = generator(noise_gen, training=True)
        fake_data_disc = generator(noise_disc, training=True)
        
        # Discriminator outputs
        real_output = discriminator(real_data, training=True)
        fake_output_disc = discriminator(fake_data_disc, training=True)
        fake_output_gen = discriminator(fake_data_gen, training=True)
        
        # Label smoothing (from swim repo)
        real_labels = smooth_positive_labels(tf.ones_like(real_output))
        fake_labels = smooth_negative_labels(tf.zeros_like(fake_output_disc))
        
        # Discriminator loss
        d_loss_real = tf.reduce_mean(
            tf.keras.losses.binary_crossentropy(real_labels, real_output))
        d_loss_fake = tf.reduce_mean(
            tf.keras.losses.binary_crossentropy(fake_labels, fake_output_disc))
        total_disc_loss = d_loss_real + d_loss_fake
        
        # Generator losses
        gen_real_labels = smooth_positive_labels(tf.ones_like(fake_output_gen))
        g_loss_adv = tf.reduce_mean(
            tf.keras.losses.binary_crossentropy(gen_real_labels, fake_output_gen))
        
        # Huber reconstruction loss (from swim repo — more robust than MSE)
        g_loss_recon = tf.reduce_mean(
            tf.keras.losses.huber(real_data, fake_data_gen, delta=0.1))
        
        # Moment matching loss (std + skewness + kurtosis)
        g_loss_moment = compute_moment_loss(real_data, fake_data_gen)
        # Tail quantile matching loss (bottom/top 5% — targets extreme values)
        g_loss_tail = compute_tail_loss(real_data, fake_data_gen, tail_pct=0.05)
        
        total_gen_loss = (LAMBDA_ADV * g_loss_adv
                          + LAMBDA_RECON * g_loss_recon
                          + LAMBDA_MOMENT * g_loss_moment
                          + LAMBDA_TAIL * g_loss_tail)
    
    # Compute gradients
    disc_grads = disc_tape.gradient(total_disc_loss, discriminator.trainable_variables)
    gen_grads = gen_tape.gradient(total_gen_loss, generator.trainable_variables)
    
    # Gradient clipping (from swim repo — clip to [-1, 1])
    disc_grads = [tf.clip_by_value(g, -1.0, 1.0) if g is not None else g 
                  for g in disc_grads]
    gen_grads = [tf.clip_by_value(g, -1.0, 1.0) if g is not None else g 
                 for g in gen_grads]
    
    # Apply gradients
    disc_optimizer.apply_gradients(
        zip(disc_grads, discriminator.trainable_variables))
    gen_optimizer.apply_gradients(
        zip(gen_grads, generator.trainable_variables))
    
    return {
        'disc_loss': total_disc_loss,
        'gen_loss': total_gen_loss,
        'disc_real': d_loss_real,
        'disc_fake': d_loss_fake,
        'gen_adv': g_loss_adv,
        'gen_recon': g_loss_recon,
        'gen_moment': g_loss_moment,
        'gen_tail': g_loss_tail,
    }


print("Training step compiled.")



TensorBoard log dir: ../logs/wavenet_gan/20260306-205409
  → Launch: tensorboard --logdir ../logs/wavenet_gan
Training step compiled.


In [ ]:
# Create tf.data.Dataset for mini-batch training
dataset = tf.data.Dataset.from_tensor_slices(sequences)
dataset = dataset.shuffle(buffer_size=len(sequences), seed=42)
dataset = dataset.batch(BATCH_SIZE, drop_remainder=True)
dataset = dataset.prefetch(tf.data.AUTOTUNE)

# Training loop
print(f"Training WaveNet GAN for {EPOCHS} epochs...")
print(f"Dataset: {len(sequences)} sequences, {len(sequences)//BATCH_SIZE} steps/epoch")
print(f"Generator LR: {LR_GENERATOR}, Discriminator LR: {LR_DISCRIMINATOR}")
print(f"Adaptive LR re-balance every {LR_ADJUST_EVERY} epochs")
print(f"Lambda: adv={LAMBDA_ADV}, recon={LAMBDA_RECON}, moment={LAMBDA_MOMENT}, tail={LAMBDA_TAIL}")
print("="*70)

history = {
    'disc_loss': [], 'gen_loss': [],
    'disc_real': [], 'disc_fake': [],
    'gen_adv': [], 'gen_recon': [], 'gen_moment': [], 'gen_tail': [],
    'lr_gen': [], 'lr_disc': []
}

for epoch in range(EPOCHS):
    epoch_metrics = {key: [] for key in history if key not in ('lr_gen', 'lr_disc')}
    
    for batch in dataset:
        batch_size = tf.shape(batch)[0]
        metrics = train_step(batch, batch_size)
        
        for key in epoch_metrics:
            epoch_metrics[key].append(metrics[key].numpy())
    
    # Record epoch averages
    for key in epoch_metrics:
        history[key].append(np.mean(epoch_metrics[key]))

    # --- Balanced Adaptive LR ---
    # Compare per-term adversarial losses only (disc_loss/2 vs gen_adv)
    # so both are single BCE terms and ratio≈1.0 at equilibrium is meaningful.
    # Using raw disc_loss vs gen_loss compares different scales (2 BCE terms vs adv+100×recon).
    if (epoch + 1) % LR_ADJUST_EVERY == 0:
        new_gen_lr, new_disc_lr = adaptive_lr(
            history['disc_loss'][-1] / 2.0,   # per-term D loss
            history['gen_adv'][-1]             # adversarial G loss only
        )
        gen_optimizer.learning_rate.assign(new_gen_lr)
        disc_optimizer.learning_rate.assign(new_disc_lr)

    history['lr_gen'].append(float(gen_optimizer.learning_rate))
    history['lr_disc'].append(float(disc_optimizer.learning_rate))

    # --- TensorBoard logging (matching NB08 metrics) ---
    with tb_writer.as_default():
        tf.summary.scalar('loss/disc_total', history['disc_loss'][-1], step=epoch)
        tf.summary.scalar('loss/disc_real', history['disc_real'][-1], step=epoch)
        tf.summary.scalar('loss/disc_fake', history['disc_fake'][-1], step=epoch)
        tf.summary.scalar('loss/gen_total', history['gen_loss'][-1], step=epoch)
        tf.summary.scalar('loss/gen_adversarial', history['gen_adv'][-1], step=epoch)
        tf.summary.scalar('loss/gen_reconstruction', history['gen_recon'][-1], step=epoch)
        tf.summary.scalar('loss/gen_moment', history['gen_moment'][-1], step=epoch)
        tf.summary.scalar('loss/gen_tail', history['gen_tail'][-1], step=epoch)
        tf.summary.scalar('lr/generator', history['lr_gen'][-1], step=epoch)
        tf.summary.scalar('lr/discriminator', history['lr_disc'][-1], step=epoch)

    if epoch % 100 == 0:
        print(f"Epoch {epoch:4d}/{EPOCHS} | "
              f"D: {history['disc_loss'][-1]:.4f} "
              f"(r:{history['disc_real'][-1]:.3f} f:{history['disc_fake'][-1]:.3f}) | "
              f"G: {history['gen_loss'][-1]:.4f} "
              f"(adv:{history['gen_adv'][-1]:.3f} rec:{history['gen_recon'][-1]:.4f} "
              f"mom:{history['gen_moment'][-1]:.3f} tail:{history['gen_tail'][-1]:.4f}) | "
              f"LR {history['lr_gen'][-1]:.1e}/{history['lr_disc'][-1]:.1e}")

tb_writer.flush()

print("\nTraining completed!")

Training WaveNet GAN for 2000 epochs...
Dataset: 6012 sequences, 46 steps/epoch
Generator LR: 0.0002, Discriminator LR: 0.0001
Adaptive LR re-balance every 50 epochs
Lambda: adv=1.0, recon=100.0, moment=1.0, tail=50.0
Epoch    0/2000 | D: 1.3774 (r:0.612 f:0.765) | G: 12.8238 (adv:0.613 rec:0.0055 mom:10.430 tail:0.0246) | LR 2.0e-04/1.0e-04
Current d_loss/g_loss ratio: 1.13
  → In equilibrium band. LRs unchanged: gen=2.00e-04, disc=1.00e-04
Current d_loss/g_loss ratio: 0.71
  → In equilibrium band. LRs unchanged: gen=2.00e-04, disc=1.00e-04
Epoch  100/2000 | D: 1.1672 (r:0.518 f:0.649) | G: 4.3109 (adv:0.865 rec:0.0022 mom:3.152 tail:0.0015) | LR 2.0e-04/1.0e-04
Current d_loss/g_loss ratio: 0.69
  → In equilibrium band. LRs unchanged: gen=2.00e-04, disc=1.00e-04
Current d_loss/g_loss ratio: 0.65
  → In equilibrium band. LRs unchanged: gen=2.00e-04, disc=1.00e-04
Epoch  200/2000 | D: 1.1532 (r:0.510 f:0.643) | G: 5.1445 (adv:0.898 rec:0.0022 mom:3.928 tail:0.0019) | LR 2.0e-04/1.0e-04


## 6. Training Loss Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Total losses
axes[0].plot(history['disc_loss'], label='Discriminator', alpha=0.7)
axes[0].plot(history['gen_loss'], label='Generator', alpha=0.7)
axes[0].set_title('Total Losses')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

# Discriminator breakdown
axes[1].plot(history['disc_real'], label='D(real)', alpha=0.7)
axes[1].plot(history['disc_fake'], label='D(fake)', alpha=0.7)
axes[1].set_title('Discriminator Losses')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

# Generator breakdown
axes[2].plot(history['gen_adv'], label='Adversarial', alpha=0.7)
axes[2].plot(history['gen_recon'], label='Reconstruction', alpha=0.7)
axes[2].plot(history['gen_moment'], label='Moment', alpha=0.7)
axes[2].plot(history['gen_tail'], label='Tail', alpha=0.7)
axes[2].set_title('Generator Losses')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Loss')
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()

## 7. Generate Synthetic Data

In [ ]:
n_samples = 100  # Generate more samples for robust evaluation
#generator = load_model('../models/wavenet_generator.keras')
# Generate synthetic sequences 
Z_synthetic = tf.random.normal(
    shape=(n_samples, SEQUENCE_LENGTH, LATENT_DIM), mean=0.0, stddev=1.0
)
synthetic_sequences = generator(Z_synthetic, training=False)

# Reshape to (n_samples * sequence_length, 1) then rescale
synthetic_flat = synthetic_sequences.numpy().reshape(-1, 1)
synthetic_rescaled = scaler.inverse_transform(synthetic_flat).reshape(n_samples, SEQUENCE_LENGTH)

# Plot synthetic vs real sequences
# Use well-separated indices (not consecutive — consecutive windows overlap by 23/24 steps)
plot_idx = np.linspace(0, len(sequences) - 1, 5, dtype=int)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Synthetic
for i in range(5):
    axes[0].plot(synthetic_rescaled[i], alpha=0.7, label=f'Synthetic {i+1}')
axes[0].set_title('WaveNet GAN — Synthetic Sequences')
axes[0].set_xlabel('Time Step')
axes[0].set_ylabel('Log Return')
axes[0].legend()
axes[0].grid(True)

# Real (evenly spaced from dataset to avoid overlapping windows)
real_plot = scaler.inverse_transform(
    sequences[plot_idx].reshape(-1, 1)
).reshape(5, SEQUENCE_LENGTH)
for i in range(5):
    axes[1].plot(real_plot[i], alpha=0.7, label=f'Real {i+1}')
axes[1].set_title('Real Sequences')
axes[1].set_xlabel('Time Step')
axes[1].set_ylabel('Log Return')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

print(f"Synthetic stats — mean: {synthetic_rescaled.mean():.6f}, "
      f"std: {synthetic_rescaled.std():.6f}, "
      f"min: {synthetic_rescaled.min():.6f}, max: {synthetic_rescaled.max():.6f}")
print(f"Real stats     — mean: {log_returns.mean():.6f}, "
      f"std: {log_returns.std():.6f}, "
      f"min: {log_returns.min():.6f}, max: {log_returns.max():.6f}")

## 8. Save Models

In [ ]:
os.makedirs('../models', exist_ok=True)

generator.save('../models/wavenet_generator_ml_tl.keras')
discriminator.save('../models/wavenet_discriminator_ml_tl.keras')

# Also save weights separately
generator.save_weights('../models/wavenet_generator_ml_tl.weights.h5')
discriminator.save_weights('../models/wavenet_discriminator_ml_tl.weights.h5')

# Save synthetic data
np.save('../data/processed/synthetic_wavenet_ml_tl.npy', synthetic_sequences.numpy())
pd.DataFrame(
    synthetic_rescaled.flatten(), columns=['Synthetic_LogReturn']
).to_csv('../data/synthetic/wavenet_synthetic_full_ml_tl.csv', index=False)

print("Models and synthetic data saved.")

## 9. Evaluation — Comparison Metrics

Compute the same metrics across both models for a fair head-to-head comparison.

Metrics from `utils/evaluation_metrics.py` (ported from swim repo's `utils_metrics.py`):
- **MMD** (Maximum Mean Discrepancy)
- **Fréchet Distance**  
- **DTW** (Dynamic Time Warping)

Plus finance-specific:
- **Autocorrelation comparison** (volatility clustering)
- **PCA / t-SNE visualization**
- **Discriminative score** (train-on-real vs synthetic, test on held-out)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from scipy.linalg import sqrtm
from scipy.spatial.distance import euclidean as scipy_euclidean
import sys
sys.path.append('..')
from utils.evaluation_metrics import compute_mmd, compute_frechet_distance

# Prepare data for comparison (3D: samples, timesteps, features)
# Use a matched number of real samples
n_eval = min(n_samples, len(sequences))
real_eval = sequences[:n_eval]           # (n_eval, 24, 1)
synth_eval = synthetic_sequences.numpy()[:n_eval]  # (n_eval, 24, 1)

# Flatten to 2D for metrics that need it
real_2d = real_eval.reshape(n_eval, SEQUENCE_LENGTH)
synth_2d = synth_eval.reshape(n_eval, SEQUENCE_LENGTH)

print(f"Evaluation sample sizes — Real: {real_eval.shape}, Synthetic: {synth_eval.shape}")

In [ ]:
# ============================================
# MMD (Maximum Mean Discrepancy)
# ============================================
mmd_score = compute_mmd(real_2d, synth_2d)
print(f"MMD (WaveNet GAN): {mmd_score:.6f}")

# ============================================
# Fréchet Distance
# ============================================
fd_results = compute_frechet_distance(real_eval, synth_eval, per_channel=True)
print(f"Fréchet Distance (overall): {fd_results['overall_fd']:.6f}")
print(f"Fréchet Distance (avg channel): {fd_results['average_channel_fd']:.6f}")
for ch, fd_val in fd_results['channel_fd'].items():
    print(f"  {ch}: {fd_val:.6f}")

In [ ]:
# ============================================
# Autocorrelation Comparison (finance-specific)
# ============================================
def compute_autocorrelation(data, max_lag=10):
    """Compute autocorrelation for each lag."""
    data_flat = data.flatten()
    mean = np.mean(data_flat)
    var = np.var(data_flat)
    acf = []
    for lag in range(1, max_lag + 1):
        cov = np.mean((data_flat[:-lag] - mean) * (data_flat[lag:] - mean))
        acf.append(cov / (var + 1e-8))
    return np.array(acf)

real_acf = compute_autocorrelation(real_2d)
synth_acf = compute_autocorrelation(synth_2d)
acf_rmse = np.sqrt(np.mean((real_acf - synth_acf)**2))

# Also compute for squared returns (volatility clustering)
real_sq_acf = compute_autocorrelation(real_2d**2)
synth_sq_acf = compute_autocorrelation(synth_2d**2)
sq_acf_rmse = np.sqrt(np.mean((real_sq_acf - synth_sq_acf)**2))

print(f"Autocorrelation RMSE (returns): {acf_rmse:.6f}")
print(f"Autocorrelation RMSE (squared returns — volatility clustering): {sq_acf_rmse:.6f}")

# Plot autocorrelations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

lags = np.arange(1, 11)
axes[0].bar(lags - 0.15, real_acf, width=0.3, label='Real', alpha=0.7)
axes[0].bar(lags + 0.15, synth_acf, width=0.3, label='Synthetic', alpha=0.7)
axes[0].set_title('Autocorrelation of Returns')
axes[0].set_xlabel('Lag')
axes[0].set_ylabel('ACF')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].bar(lags - 0.15, real_sq_acf, width=0.3, label='Real', alpha=0.7)
axes[1].bar(lags + 0.15, synth_sq_acf, width=0.3, label='Synthetic', alpha=0.7)
axes[1].set_title('Autocorrelation of Squared Returns (Volatility Clustering)')
axes[1].set_xlabel('Lag')
axes[1].set_ylabel('ACF')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# PCA and t-SNE Visualization
# ============================================
combined = np.vstack([real_2d, synth_2d])
labels = np.array([0] * n_eval + [1] * n_eval)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(combined)
axes[0].scatter(pca_result[labels==0, 0], pca_result[labels==0, 1],
                alpha=0.4, label='Real', s=10)
axes[0].scatter(pca_result[labels==1, 0], pca_result[labels==1, 1],
                alpha=0.4, label='Synthetic (WaveNet)', s=10)
axes[0].set_title('PCA: Real vs WaveNet Synthetic')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# t-SNE
# Subsample for t-SNE speed
n_tsne = min(500, n_eval)
idx_real = np.random.choice(n_eval, n_tsne, replace=False)
idx_synth = np.random.choice(n_eval, n_tsne, replace=False) + n_eval
idx_all = np.concatenate([idx_real, idx_synth])

tsne = TSNE(n_components=2, perplexity=30, random_state=42)
tsne_result = tsne.fit_transform(combined[idx_all])
tsne_labels = labels[idx_all]

axes[1].scatter(tsne_result[tsne_labels==0, 0], tsne_result[tsne_labels==0, 1],
                alpha=0.4, label='Real', s=10)
axes[1].scatter(tsne_result[tsne_labels==1, 0], tsne_result[tsne_labels==1, 1],
                alpha=0.4, label='Synthetic (WaveNet)', s=10)
axes[1].set_title('t-SNE: Real vs WaveNet Synthetic')
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
sample_size = 250
idx = np.random.permutation(len(real_eval))[:sample_size]

In [ ]:
# Data preprocessing
real_sample = np.asarray(real_eval)[idx]
synthetic_sample = np.asarray(synth_eval)[idx]

In [ ]:
real_sample_2d = real_sample.reshape(-1, SEQUENCE_LENGTH)
synthetic_sample_2d = synthetic_sample.reshape(-1, SEQUENCE_LENGTH)

In [ ]:
real_sample_2d.shape, synthetic_sample_2d.shape

In [ ]:
pca = PCA(n_components=2)
pca.fit(real_sample_2d)

pca_real = (pd.DataFrame(pca.transform(real_sample_2d))
            .assign(Data='Real'))
pca_synthetic = (pd.DataFrame(pca.transform(synthetic_sample_2d))
                 .assign(Data='Synthetic'))
pca_result = pd.concat([pca_real, pca_synthetic]).rename(
    columns={0: '1st Component', 1: '2nd Component'})

In [ ]:
tsne_data = np.concatenate((real_sample_2d,
                            synthetic_sample_2d), axis=0)

tsne = TSNE(n_components=2,
            verbose=1,
            perplexity=40)
tsne_result = tsne.fit_transform(tsne_data)

In [ ]:
tsne_result = pd.DataFrame(tsne_result, columns=['X', 'Y']).assign(Data='Real')
tsne_result.loc[sample_size*1:, 'Data'] = 'Synthetic'

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(14, 5))

sns.scatterplot(x='1st Component', y='2nd Component', data=pca_result,
                hue='Data', style='Data', ax=axes[0])
sns.despine()
axes[0].set_title('PCA Result')


sns.scatterplot(x='X', y='Y',
                data=tsne_result,
                hue='Data', 
                style='Data', 
                ax=axes[1])
sns.despine()
for i in [0, 1]:
    axes[i].set_xticks([])
    axes[i].set_yticks([])

axes[1].set_title('t-SNE Result')
fig.suptitle('Assessing Diversity: Qualitative Comparison of Real and Synthetic Data Distributions', 
             fontsize=14)
fig.tight_layout()
fig.subplots_adjust(top=.88);

In [ ]:
# ============================================
# Distribution Comparison
# ============================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of values
axes[0].hist(real_2d.flatten(), bins=80, alpha=0.5, density=True, label='Real')
axes[0].hist(synth_2d.flatten(), bins=80, alpha=0.5, density=True, label='Synthetic')
axes[0].set_title('Distribution of Scaled Log Returns')
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Density')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# QQ-style comparison (sorted values)
real_sorted = np.sort(real_2d.flatten())
synth_sorted = np.sort(synth_2d.flatten())
# Subsample to same length if needed
n_qq = min(len(real_sorted), len(synth_sorted))
axes[1].scatter(
    real_sorted[np.linspace(0, len(real_sorted)-1, n_qq, dtype=int)],
    synth_sorted[np.linspace(0, len(synth_sorted)-1, n_qq, dtype=int)],
    alpha=0.3, s=5
)
axes[1].plot([0, 1], [0, 1], 'r--', label='Perfect match')
axes[1].set_title('Q-Q Plot: Real vs Synthetic')
axes[1].set_xlabel('Real Quantiles')
axes[1].set_ylabel('Synthetic Quantiles')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# Discriminative Score
# (Train-on-synthetic, test-on-real — from original TimeGAN paper)
# ============================================
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

# Prepare labeled dataset
X_combined = np.vstack([real_2d, synth_2d])
y_combined = np.concatenate([np.ones(n_eval), np.zeros(n_eval)])

X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y_combined, test_size=0.2, random_state=42, stratify=y_combined
)

clf = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300, random_state=42)
clf.fit(X_train, y_train)
disc_accuracy = accuracy_score(y_test, clf.predict(X_test))

# Ideal discriminative score = 0.5 (can't tell real from fake)
disc_score = abs(0.5 - disc_accuracy)
print(f"Discriminative Accuracy: {disc_accuracy:.4f}")
print(f"Discriminative Score: {disc_score:.4f} (closer to 0 = better, real/fake indistinguishable)")

In [ ]:
# ============================================
# Summary Metrics Table
# ============================================
print("=" * 60)
print("WaveNet GAN — Evaluation Summary")
print("=" * 60)
print(f"{'Metric':<45} {'Score':>10}")
print("-" * 60)
print(f"{'MMD (lower=better)':<45} {mmd_score:>10.6f}")
print(f"{'Fréchet Distance (lower=better)':<45} {fd_results['overall_fd']:>10.6f}")
print(f"{'ACF RMSE — returns (lower=better)':<45} {acf_rmse:>10.6f}")
print(f"{'ACF RMSE — squared returns (lower=better)':<45} {sq_acf_rmse:>10.6f}")
print(f"{'Discriminative Score (closer to 0=better)':<45} {disc_score:>10.4f}")
print(f"{'Discriminative Accuracy (closer to 0.5=better)':<45} {disc_accuracy:>10.4f}")
print("=" * 60)
print("\nCompare these values against LSTM TimeGAN (notebook 03)")
print("and the TF2 TimeGAN reference (notebook you're adding).")